In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd

# -----------------------------
# Dataset class
# -----------------------------
class AudioEditingDataset(Dataset):
    def __init__(self, csv_file):
        self.data = pd.read_csv(csv_file)
        # Convert embeddings from string to tensors if stored as comma-separated strings
        self.data['prompt'] = self.data['embeddings_prompt1'].apply(lambda x: torch.tensor([float(i) for i in x.split(',')]))
        self.data['input_audio'] = self.data['embeddings_input_audio_1'].apply(lambda x: torch.tensor([float(i) for i in x.split(',')]))
        self.data['output_audio'] = self.data['embeddings_output_audio_1'].apply(lambda x: torch.tensor([float(i) for i in x.split(',')]))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        prompt = self.data.iloc[idx]['prompt']
        input_audio = self.data.iloc[idx]['input_audio']
        output_audio = self.data.iloc[idx]['output_audio']
        return prompt, input_audio, output_audio

# -----------------------------
# Transformer-based model
# -----------------------------
class AudioEditingTransformer(nn.Module):
    def __init__(self, embedding_dim=512, num_layers=4, num_heads=8, hidden_dim=1024):
        super().__init__()
        self.embedding_dim = embedding_dim
        # Positional encoding not strictly needed for single-vector input, but can help slightly
        self.positional_encoding = nn.Parameter(torch.zeros(1, 2, embedding_dim))
        # Transformer encoder layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            activation='relu',
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        # Final linear layer to predict output audio embedding
        self.fc_out = nn.Linear(embedding_dim, embedding_dim)
    
    def forward(self, prompt_embedding, input_audio_embedding):
        # Concatenate embeddings into a sequence of length 2
        x = torch.stack([prompt_embedding, input_audio_embedding], dim=1)  # (batch_size, seq_len=2, embedding_dim)
        x = x + self.positional_encoding  # Add positional info
        x = self.transformer(x)  # Pass through transformer encoder
        output_embedding = self.fc_out(x[:, -1, :])  # Use last token (input_audio) to predict output
        return output_embedding

# -----------------------------
# Training loop
# -----------------------------
def train_model(csv_file, embedding_dim=512, batch_size=64, epochs=20, lr=1e-4, device='cuda'):
    dataset = AudioEditingDataset(csv_file)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = AudioEditingTransformer(embedding_dim=embedding_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion_mse = nn.MSELoss()
    
    for epoch in range(epochs):
        total_loss = 0
        for prompt, input_audio, output_audio in dataloader:
            prompt = prompt.to(device).float()
            input_audio = input_audio.to(device).float()
            output_audio = output_audio.to(device).float()
            
            optimizer.zero_grad()
            pred = model(prompt, input_audio)
            # Combine MSE with cosine similarity
            loss_mse = criterion_mse(pred, output_audio)
            cos = nn.functional.cosine_similarity(pred, output_audio, dim=-1)
            loss_cos = 1 - cos.mean()
            loss = loss_mse + 0.1 * loss_cos  # Weighted sum
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item() * prompt.size(0)
        
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataset):.6f}")
    
    return model

# -----------------------------
# Example inference
# -----------------------------
def infer(model, prompt_embedding, input_audio_embedding, device='cuda'):
    model.eval()
    with torch.no_grad():
        prompt_embedding = prompt_embedding.to(device).float().unsqueeze(0)
        input_audio_embedding = input_audio_embedding.to(device).float().unsqueeze(0)
        pred_embedding = model(prompt_embedding, input_audio_embedding)
    return pred_embedding.squeeze(0)  # Return embedding vector
